In [ ]:
!pip install openai-whisper

In [ ]:
video_url = input()

In [ ]:
!pip install -q yt-dlp
!apt-get install -y ffmpeg

In [ ]:
import yt_dlp

def download_and_get_title(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'videoplayback.%(ext)s',  # save using title as filename
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)  # downloads + returns metadata
        title = info.get('title', 'unknown_title')
        print(f"Downloaded: {title}.mp3")
        return title


In [ ]:
title = download_and_get_title(video_url)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
import whisper

# Explicitly set the device when loading the model
model = whisper.load_model("small", device=device)

In [ ]:
# Ensure the model is on the correct device before transcription
model.to(device)

audio = whisper.load_audio("videoplayback.mp3")
audio = whisper.pad_or_trim(audio)
mel = whisper.log_mel_spectrogram(audio, n_mels=model.dims.n_mels).to(model.device)
_, probs = model.detect_language(mel)
languages = max(probs, key=probs.get)

result = model.transcribe("videoplayback.mp3", task="translate", language=languages)

In [ ]:
transcript_data = []
for segment in result['segments']:
  transcript_data.append({
      'id' : segment['id'],
      'start' : round(segment['start'],2),
      'end' : round(segment['end'],2),
      'text' : segment['text']
  })

In [ ]:
import pandas as pd
transcript_data = pd.DataFrame(transcript_data)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
import re

filler_words = {
    "um", "uh", "hmm", "erm", "er", "ah", "oh", "like", "you know",
    "i mean", "sort of", "kind of", "actually", "basically",
    "literally", "right", "okay", "well", "so", "just", "you see",
    "you know what I mean", "you get me", "you feel me", "let's see",
    "I guess", "alright", "kinda", "yeah", "mmm", "I suppose"
}
filler_phrases = []
for word in filler_words:
  filler_phrases.append(r"\b"+ re.escape(word) + r"\b")

In [ ]:
def remove_adj(text):
  doc = nlp(text)
  keep = []
  for token in doc:
    if token.pos_ == "INTJ":
      continue
    if token.lower_ in {"like","literally","basically"}:
      continue
    keep.append(token.text_with_ws)

  return " ".join(keep)


In [ ]:
def remove_fillers(text):
  for pattern in filler_phrases:
      text = re.sub(pattern,"",text,flags=re.IGNORECASE)
  return text.strip()

In [ ]:
transcript_data['text'] = transcript_data['text'].apply(remove_fillers)
transcript_data['text'] = transcript_data['text'].apply(remove_adj)

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-base-v2")

In [ ]:
transcript_data['text-count'] = transcript_data['text'].apply(lambda x: len(x.split()))
transcript_data['token-count'] = transcript_data['text'].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=False))
)

In [ ]:
i = 0
index = 0
MAX_TOKENS = 50000
transcript_data_after_chunking_together = []

while index < len(transcript_data):
    sum_tokens = 0
    new_text = ""
    start_time = str(int(transcript_data.iloc[index]['start'] // 60)).zfill(2)+":"+str(int(transcript_data.iloc[index]['start']%60)).zfill(2)
    # start_time = transcript_data.iloc[index]['start']
    end_time = None

    chunk_texts = []

    while index < len(transcript_data) and sum_tokens + transcript_data.iloc[index]['token-count'] <= MAX_TOKENS:
        sum_tokens += transcript_data.iloc[index]['token-count']
        chunk_texts.append(transcript_data.iloc[index]['text'].strip())
        end_time = transcript_data.iloc[index]['end']
        index += 1

    new_text = " ".join(chunk_texts)


    transcript_data_after_chunking_together.append({
        'id': i,
        'start': start_time,
        'end': end_time,
        'text': new_text
    })
    i += 1

In [ ]:
transcript_data_after_chunking_together = pd.DataFrame(transcript_data_after_chunking_together)

In [ ]:
transcript_data_after_chunking_together

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device for SentenceTransformer model: {device}")
model = SentenceTransformer('intfloat/e5-base-v2').to(device)
print(f"SentenceTransformer model is on device: {model.device}")

In [ ]:
embeddings = []

for _,element in transcript_data_after_chunking_together.iterrows():
  input_text = "passage: "+ element['text']
  print(f"Processing chunk with text: {input_text[:50]}...") # Print part of the text being processed
  embedding = model.encode(input_text,
                           batch_size = 32,
                           normalize_embeddings=True,
                           convert_to_tensor=True) # Convert to tensor here
  print(f"Embedding tensor is on device: {embedding.device}") # Print device of the embedding tensor
  embeddings.append(embedding.cpu().numpy()) # Move back to CPU and convert to numpy

transcript_data_after_chunking_together['embedding'] = embeddings

In [ ]:
import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dict_of_chunks = transcript_data_after_chunking_together.to_dict(orient = "records")

embeddings_torch = torch.tensor(np.array(transcript_data_after_chunking_together["embedding"].tolist()), dtype=torch.float32).to(device)